In this notebook, we perform grid point wise hypothesis tests. We compare normal and hot conditions by computing the (sign assigned) earth mover's distance. The distribution under the null hypothesis, i.e. hot conditions heights are smaller than or equal to normal conditions heights, is estimated using the stationary bootstrap.

In [ ]:
using Dates
using NCDatasets, DimensionalData
using Statistics, Distributions, OptimalTransport
using HDF5
using ProgressMeter, ArgCheck

In [ ]:
p = 0.1 #block length parameter
B = 1000; #number of bootstrap samples

# Functions

In [ ]:
"""
    pseudotimeseries(timeseries::Vector{Floa32}, p::Float64)

Generate a pseudo time series by resampling `timeseries` in the sense of the
"stationary bootstrap"-method [1]. Here, `p` is the geometric distribution
parameter controlling the block length generation.

[1] Politis, Dimitris N., and Joseph P. Romano. "The stationary bootstrap." 
Journal of the American Statistical association 89.428 (1994): 1303-1313.
"""
function pseudotimeseries(timeseries::Vector{Float32}, p::Float64)
    N = length(timeseries)
    n = 0 #current length of pseudo sample

    # Set-up distributions:
    geom_dist = Geometric(p)
    uni_dist = DiscreteUniform(1, N)

    # Iteratively draw blocks from `timeseries`:
    pseudo_ts = fill(NaN32, N)
    while n < N
        I = rand(uni_dist) #block starting point
        L = rand(geom_dist) #block length
        while L < 1 #but we want at least one item
            L = rand(geom_dist)
        end
        
        if n+L > N #prevent overflow in the last assignment
            L = N - n
        end
        
        if I+L <= N #authors prescripe cyclic boundary condition
            pseudo_ts[n+1:n+L] = timeseries[I:I+(L-1)]
        else
            block = timeseries[I:end]
            append!(block , timeseries[1:(I+(L-1))%N])
            pseudo_ts[n+1:n+L] = block
        end
        
        n += L
    end

    return pseudo_ts
end


"""
    sampleEMD(sample_1::Vector{Float32}, sample_2::Vector{Float32})

Computes the earth movers distance between samples `sample_1` and `sample_2`. 
"""

function sampleEMD(sample_1::Vector{Float32}, sample_2::Vector{Float32})
    dist_1 = DiscreteNonParametric([val for val in unique(sample_1)], 
                                [count(==(val), sample_1) for val in unique(sample_1)] ./ length(sample_1))
    dist_2 = DiscreteNonParametric([val for val in unique(sample_2)], 
                                [count(==(val), sample_2) for val in unique(sample_2)] ./ length(sample_2))

    return ot_cost((x, y) -> abs(x - y), dist_1, dist_2)
end


"""
    statbootstrap(timeseries::Vector{Float32}, B::Int64, p::Float64, hw::BitVector, nc::BitVector)

Performs the stationary bootstrap for given time series `timeseries` with `B` bootstrap samples,
block length generation parameter `p` taking valus in (0,1) and masks for heat wave `hw` and normal
conditions `nc`.
"""
function statbootstrap(timeseries::Vector{Float32}, B::Int64, p::Float64, hw::BitVector, nc::BitVector)
    bootstrap_statistics = zeros(Float32, B)

    for b in 1:B
        pseudo_sample = pseudotimeseries(timeseries, p)
        bootstrap_statistics[b] = sign(mean(pseudo_sample[hw]) - mean(pseudo_sample[nc])) * sampleEMD(pseudo_sample[hw], pseudo_sample[nc])
    end

    return bootstrap_statistics
end;

# REA6 

Load data

In [ ]:
# Heat wave definition:
NCDataset("./data/hw_gridded_rea6.nc", "r") do ds
    lons = ds["lon"][30:end,50:end]
    lats = ds["lat"][30:end,50:end]
    time = ds["time"][:]
    hw_ind = ds["hw_ind"][30:end,50:end,:]

    n_x, n_y = size(lats)
    n_t = length(time)

    da_hw = DimArray(isfinite.(hw_ind), (X(1:n_x), Y(1:n_y), Ti(time)), name="hw") #no HW if hw_ind is NaN (missing in Julia)
    
    global HW = da_hw
end

# Boundary layer height:
path_blh = "./data/ifs_blh_dmax_rea6.nc" # replace vt with ifs for the other estimation method

NCDataset(path_blh, "r") do ds
    lons = ds["lon"][30:end,50:end]
    lats = ds["lat"][30:end,50:end]
    time = ds["time"][:]
    blh = ds["blh"][30:end,50:end,:]

    n_x, n_y = size(lats)
    n_t = length(time)

    da_blh = DimArray(blh, (X(1:n_x), Y(1:n_y), Ti(time)), name="blh")
    
    global BLH = da_blh
end

# Align time steps:
common_time = intersect(lookup(BLH, Ti), lookup(HW, Ti))
HW = HW[Ti(At(common_time))]
BLH = BLH[Ti(At(common_time))];

Perform bootstrap

In [ ]:
p_vals = fill(NaN, (size(HW,1), size(HW,2)))

progbar = Progress(size(HW,1))
Threads.@threads for i_x in 1:size(HW, 1)
    for i_y in 1:size(HW, 2)
        if all(BLH[X(i_x), Y(i_y)] .=== missing) 
            continue #missing is treated different from NaN and can not be catched by e.g. isfinite
        elseif all(HW[X(i_x), Y(i_y)] .=== missing)
            continue
        elseif sum(skipmissing(HW[X(i_x), Y(i_y)])) <= 3
            continue
        else
            # Get HW indices:
            hw = HW[X(i_x), Y(i_y), Ti(At(lookup(BLH, Ti).data))].data .=== true
            nc = .!hw

            timeseries = convert(Vector{Float32}, BLH[X(i_x), Y(i_y)].data)

            # Perform bootstrap
            sample_statistic = sign(mean(timeseries[hw]) - mean(timeseries[nc])) * sampleEMD(timeseries[hw], timeseries[nc])
            bootstrap_statistics = statbootstrap(timeseries, B, p, hw, nc)
            p_vals[i_x,i_y] = 1. - sum(bootstrap_statistics .<= sample_statistic) / B
        end
    end
    next!(progbar)
end
finish!(progbar)

h5open("./data/emd_rea6_ifs_blh.hdf5", "w") do file
    file["p"] = p_vals
end;

# ERA5

Load data

In [ ]:
# Heat wave definition:
NCDataset("./data/hw_gridded_era5.nc", "r") do ds
    lons = ds["longitude"][:]
    lats = ds["latitude"][:]
    time = ds["time"][:]
    hw_ind = ds["hw_ind"][:,:,:]

    n_x, n_y = length(lons), length(lats)
    n_t = length(time)

    da_hw = DimArray(isfinite.(hw_ind), (X(1:n_x), Y(1:n_y), Ti(time)), name="hw")
    
    global HW = da_hw
end

# Boundary layer height:
path_blh = "./data/blh_dmax_era5.nc"

NCDataset(path_blh, "r") do ds
    lons = ds["longitude"][:]
    lats = ds["latitude"][:]
    time = ds["valid_time"][:] .+ Hour(1)
    blh = ds["blh"][:,:,:]

    n_x, n_y = length(lons), length(lats)
    n_t = length(time)

    da_blh = DimArray(blh, (X(1:n_x), Y(1:n_y), Ti(time)), name="blh")
    
    global BLH = da_blh
end;

# Align time steps:
common_time = intersect(lookup(BLH, Ti), lookup(HW, Ti))
common_time = filter(dt -> 2014 <= year(dt) <= 2018, common_time) # narrow down time frame, skip line for full timeseries
HW = HW[Ti(At(common_time))]
BLH = BLH[Ti(At(common_time))];

Perform bootstrap

In [ ]:
p_vals = fill(NaN, (size(HW,1), size(HW,2)))

progbar = Progress(size(HW,1)*size(HW,2))
Threads.@threads for i_x in 1:size(HW, 1)
    for i_y in 1:size(HW, 2)
        if all(BLH[X(i_x), Y(i_y)] .=== missing) 
            continue #missing is treated different from NaN and can not be catched by e.g. isfinite
        elseif all(HW[X(i_x), Y(i_y)] .=== missing)
            continue
        elseif sum(skipmissing(HW[X(i_x), Y(i_y)])) <= 3
            continue
        else
            # Get HW indices:
            hw = HW[X(i_x), Y(i_y), Ti(At(lookup(BLH, Ti).data))].data .=== true
            nc = .!hw

            timeseries = convert(Vector{Float32}, BLH[X(i_x), Y(i_y)].data)

            # Perform bootstrap
            sample_statistic = sign(mean(timeseries[hw]) - mean(timeseries[nc])) * sampleEMD(timeseries[hw], timeseries[nc])
            bootstrap_statistics = statbootstrap(timeseries, B, p, hw, nc)
            p_vals[i_x,i_y] = 1. - sum(bootstrap_statistics .<= sample_statistic) / B
        end
        next!(progbar)
    end
end
finish!(progbar)

h5open("./data/emd_era5_14to18.hdf5", "w") do file
    file["p"] = p_vals
end;